<a href="https://colab.research.google.com/github/krishnendu1233/A2A/blob/main/Analysis_of_100k_tm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from huggingface_hub import notebook_login, hf_hub_download

notebook_login()

In [3]:
#@title Download model

hf_hub_download(
repo_id="Krudev/kudos",
filename="thumbnail_title/TubeCLIP.ckpt",
local_dir="./"
)#thumbnail_title/TubeCLIP.ckpt

thumbnail_title/TubeCLIP.ckpt: reconstructing file:   0%|          |  0.00B / 1.96GB            

thumbnail_title/TubeCLIP.ckpt: downloading bytes:           |  0.00B            

'/content/thumbnail_title/TubeCLIP.ckpt'

In [4]:
#@title ###Configurations

import json
import torch
import os
import requests
import argparse
from predict import CLIPForViewsClassification, NUM_CLASSES, CLIP_MODEL_NAME, LORA_R, LORA_ALPHA, LORA_DROPOUT, LORA_TARGET_MODULES, LEARNING_RATE, FEATURE_COMBINATION_STRATEGY
# For model and data
INPUT_FILE = "/content/drive/MyDrive/100k_analysis/stratified_dataset.jsonl"#/content/drive/MyDrive/100k_analysis/stratified_dataset.jsonl
OUTPUT_FILE = "results.jsonl"
TEMP_DIR = "temp_thumbnails"
MODEL_PATH="//content/thumbnail_title/TubeCLIP.ckpt"

# For main loop
BATCH_SIZE=1000
NUM_LOOP=10
# 10000

In [ ]:
from google.colab import files
files.upload()

In [17]:
#!pip install pytorch-lightning
!pip install torchao==0.16.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 32.2 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [3]:
#@title #### Main **Analysis** loop func
def run_pipeline(num_loops, batch_size=10):
    """
    Loops functions 'a' and 'b' a specific number of times.
    """
    model = load_model()

    # Auto-detect last_id from the existing output file (if resuming later)
    last_id = None
    if os.path.exists(OUTPUT_FILE):
        with open(OUTPUT_FILE, 'r', encoding='utf-8') as f:
            lines = f.readlines()
            if lines:
                last_record = json.loads(lines[-1])
                last_id = last_record.get("video_id")

    for i in range(num_loops):
        print(f"\n--- Loop {i+1}/{num_loops} | Last ID: {last_id} ---")

        # Step A: Download batch
        print(f"Downloading next {batch_size} thumbnails...")
        batch = download_batch(last_id, batch_size)

        if not batch:
            print("No more videos found to process. Pipeline finished.")
            break

        # Step B: Process and cleanup
        print(f"Processing {len(batch)} items and cleaning up...")
        process_and_cleanup_batch(batch, model)

        # Update last_id for the next loop
        last_id = batch[-1]['video_id']
        print(f"Batch complete. New last_id: {last_id}")

In [8]:
print(model)

CLIPForViewsClassification(
  (clip_model): CLIPModel(
    (text_model): PeftModel(
      (base_model): LoraModel(
        (model): CLIPTextModel(
          (embeddings): CLIPTextEmbeddings(
            (token_embedding): Embedding(49408, 768)
            (position_embedding): Embedding(77, 768)
          )
          (encoder): CLIPEncoder(
            (layers): ModuleList(
              (0-11): 12 x CLIPEncoderLayer(
                (self_attn): CLIPAttention(
                  (k_proj): lora.Linear(
                    (base_layer): Linear(in_features=768, out_features=768, bias=True)
                    (lora_dropout): ModuleDict(
                      (default): Dropout(p=0.1, inplace=False)
                    )
                    (lora_A): ModuleDict(
                      (default): Linear(in_features=768, out_features=32, bias=False)
                    )
                    (lora_B): ModuleDict(
                      (default): Linear(in_features=32, out_features=768, bias=Fa

In [4]:
model = load_model()

Loading model...


ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
    ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_13258/2496151566.py", line 1, in <cell line: 0>
    model = load_model()
  File "/tmp/ipykernel_13258/2907254299.py", line 10, in load_model
    model = CLIPForViewsClassification.load_from_checkpoint(
            MODEL_PATH,
    ...<8 lines>...
            num_training_steps_total=1000 # Placeholder, not used for inference
        )
  File "/usr/local/lib/python3.13/dist-packages/pytorch_lightning/utilities/model_helpers.py", line 130, in wrapper
    return self.method(cls_type, *args, **kwargs)
           ~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/pytorch_lightning/core/module.py", line 1796, in load_from_checkpoint
    loaded = _load_from_checkpoint(
        cls,
    ...<5

TypeError: object of type 'NoneType' has no len()

In [13]:
# test run the loop
run_pipeline(num_loops=1, batch_size=10)

Loading model...


Loading weights:   0%|          | 0/590 [00:00<?, ?it/s]


--- Loop 1/1 | Last ID: FXqeW-zZFeU ---
Processing 10 items and cleaning up...
Batch complete. New last_id: -NrD1kXTbDs


In [ ]:
import os
os.kill(os.getpid(),9)

In [12]:

#@title ####Utilities - *to help analyse*
# Model Loader
def load_model():
    """
    Function to load the prediction model.
    Replace this with your actual model loading logic (e.g., PyTorch, TensorFlow).
    """
    print("Loading model...")

    model = CLIPForViewsClassification.load_from_checkpoint(
            MODEL_PATH,
            num_classes=NUM_CLASSES,
            clip_model_name=CLIP_MODEL_NAME,
            lora_r=LORA_R,
            lora_alpha=LORA_ALPHA,
            lora_dropout=LORA_DROPOUT,
            lora_target_modules=LORA_TARGET_MODULES,
            learning_rate=LEARNING_RATE,
            feature_combination_strategy=FEATURE_COMBINATION_STRATEGY,
            num_training_steps_total=1000 # Placeholder, not used for inference
        )
    model.eval()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
   # except exception as e:
        #print(f"{e}")

    return model

# Reused Predictor
def predict(model, thumbnail_path, title):
    """
    Function to run predictions.
    """
    # Mock prediction logic
    result = model.predict_step(image_path=thumbnail_path, title=title)

    return result

# Batch Downloader (working correctly - checked)
def download_batch(last_id, batch_size=10):
    """
    Reads the input file, skips until last_id is found,
    and downloads the next batch of thumbnails.
    """
    if not os.path.exists(TEMP_DIR):
        os.makedirs(TEMP_DIR)

    batch_data = []

    # If last_id is None or empty, start from the beginning
    found_last = False if last_id else True

    try:
        with open(INPUT_FILE, 'r', encoding='utf-8') as f:
            for line in f:
                if not line.strip():
                    continue

                data = json.loads(line)
                vid_id = data.get('video_id')
                title = data.get('title')

                # Skip records until we find the last_id processed
                if not found_last:
                    if vid_id == last_id:
                        found_last = True
                    continue

                # Construct YouTube thumbnail URL
                # hqdefault.jpg is a reliable high-quality thumbnail endpoint
                thumbnail_url = f"https://img.youtube.com/vi/{vid_id}/hqdefault.jpg"
                filepath = os.path.join(TEMP_DIR, f"{vid_id}.jpg")

                # Download thumbnail
                try:
                    response = requests.get(thumbnail_url, timeout=10)
                    if response.status_code == 200:
                        with open(filepath, 'wb') as img_f:
                            img_f.write(response.content)

                        # Add metadata for the processor function
                        batch_data.append({
                            "video_id": vid_id,
                            "title": title,
                            "thumbnail_path": filepath,
                            "video_link": f"https://www.youtube.com/watch?v={vid_id}"
                        })

                        # Stop if we hit our batch limit
                        if len(batch_data) >= batch_size:
                            break
                    else:
                        print(f"Warning: Failed to fetch thumbnail for {vid_id} (Status: {response.status_code})")
                except requests.RequestException as e:
                    print(f"Error downloading {vid_id}: {e}")

    except FileNotFoundError:
        print(f"Error: Input file '{INPUT_FILE}' not found.")

    return batch_data

# Cleaner
def process_and_cleanup_batch(batch_data, model):
    """
    Runs prediction on the batch, appends results to JSONL,
    and deletes the thumbnail files.
    """
    # Open in 'a' (append) mode
    with open(OUTPUT_FILE, 'a', encoding='utf-8') as out_f:
        for item in batch_data:
            # 1. Predict
            pred_result = predict(model, item['thumbnail_path'], item['title'])

            # 2. Append to output file
            result_record = {
                "video_id": item['video_id'],
                "title": item['title'],
                "video_link": item['video_link'],
                "prediction": { "predicted_class": pred_result['predicted_class'],
                                "probabilities": pred_result['probabilities'].tolist()
            }}
            out_f.write(json.dumps(result_record) + '\n')

            # 3. Cleanup thumbnail
            if os.path.exists(item['thumbnail_path']):
                os.remove(item['thumbnail_path'])